In [0]:
# =========================================================================
# GOLD LAYER: dim_calendar Worker Notebook
# =========================================================================

GOLD_STORAGE_PATH = "abfss://gold@aze2estorage.dfs.core.windows.net/dimensions/dim_calendar"
TABLE_NAME = "gold.dim_calendar"

print(f"🔄 Starting build for {TABLE_NAME}...")
print(f"📂 Target Storage Path: {GOLD_STORAGE_PATH}")

# Read from Silver Calendar
df_calendar = spark.read.format("delta").load("abfss://silver@aze2estorage.dfs.core.windows.net/calendar")

# Select schema for BI readiness
gold_dim_calendar = df_calendar.select(
    "Date",
    "year",
    "month",
    "quarter",
    "day_of_week"
)

# Write to explicit Gold storage path
gold_dim_calendar.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(GOLD_STORAGE_PATH)

# Register external table in Unity Catalog
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE_NAME}
    USING DELTA
    LOCATION '{GOLD_STORAGE_PATH}'
""")

# Optimize for date-based slicing in Power BI
spark.sql(f"OPTIMIZE {TABLE_NAME} ZORDER BY (Date)")

print(f"🎉 Successfully built and registered {TABLE_NAME} ({gold_dim_calendar.count()} records)")